# Dependencies

In [58]:
import re
import pandas as pd
import plotly.express as px
from itertools import product

import warnings
warnings.filterwarnings("ignore")

# Load dataset

In [7]:
fpath = "../data/20250630_SCENES_metadata_integrated_modified.xlsx"
meta_integrated = pd.read_excel(fpath)

meta_df = meta_integrated.copy()

In [8]:
def extract_parentheses_content(scene):
    scene = str(scene).strip()
    match = re.search(r'\((.*?)\)', scene)
    if match:
        return match.group(1).strip()
    else:
        return ""
    
if "scene" in meta_df:
    meta_df["scene"] = meta_df["scene"].apply(extract_parentheses_content)
if "lighting" in meta_df:
    meta_df["lighting"] = meta_df["lighting"].apply(extract_parentheses_content)


In [19]:
meta_df.columns

Index(['record', 'project_id', 'project_name', 'project_start', 'data_curator',
       'project_record', 'setup', 'device_geometry', 'measurement_id',
       'record_id', 'date', 'time', 'protocol', 'experimenter',
       'available_$device', 'available_gloptic_spectis1',
       'available_gigahertz-optic_msc15', 'available_jeti_spectraval1511',
       'available_westborophotonics_wp690e-circ', 'wp690csv',
       'available_kleininstruments_k-10a', 'available_inteldrealsense_d455',
       'available_gopro_hero10black_video',
       'available_gopro_hero10black_photo', 'location', 'latitude',
       'longitude', 'direction', 'elevation', 'groundheight', 'boxtilt',
       'boxheight', 'temperature', 'humidity', 'weather', 'other_weather',
       'wind', 'scene', 'other_scene', 'view', 'other_view', 'lighting',
       'other_lighting', 'technotes', 'available_all', 'hour'],
      dtype='object')

In [22]:
VIEW_COLORS = {
    "Outdoor": "#CD923B",
    "Indoor w/ window": "#081A5B",
    "Indoor w/o window":  "#426E56"
}

In [38]:
meta_df['hour'] = meta_df['time'].astype(str).str.zfill(4).str[:2].astype(int)
meta_df['date'] = pd.to_datetime(meta_df['date'], format="%Y%m%d")
meta_df["date"] = pd.to_datetime(meta_df["date"], errors="coerce").dt.date
meta_df = meta_df.sort_values("date")


In [128]:
counts = (
    meta_df.dropna(subset=["date","view"])
           .groupby(["date","view"]).size()
           .unstack("view", fill_value=0)
)

full_idx = pd.date_range(counts.index.min(), counts.index.max(), freq="D").date
counts = counts.reindex(full_idx, fill_value=0)
counts.index.name = "date"

long_df = counts.reset_index().melt(id_vars="date", var_name="view", value_name="count")
long_df["date"] = pd.to_datetime(long_df["date"])

fig = px.bar(
    long_df,
    x="date", y="count", color="view",
    color_discrete_map=VIEW_COLORS,
    title="Indoor vs Outdoor by date"
)

fig.update_layout(
    bargap=0.01,
    barmode="stack",
    xaxis_title=None,
    yaxis_title="Count",
    legend_title="",
    legend=dict(
        orientation="h",       
        yanchor="bottom",      
        y=1.02,                
        xanchor="center",      
        x=0.5,
    ),
    height=500,
)

fig.update_xaxes(tickangle=305)
fig.show()

fig.write_html("../assets/plots/daily_stacked_by_view.html", include_plotlyjs="cdn", full_html=False)

In [119]:

meta_df = meta_df.copy()
meta_df["hour"] = pd.to_numeric(meta_df["hour"], errors="coerce").astype("Int64")
meta_df = meta_df.dropna(subset=["hour", "view"])
meta_df["hour"] = meta_df["hour"].astype(int).clip(0, 23)

hour_counts = (
    meta_df.groupby(["hour", "view"])
           .size()
           .reset_index(name="count")
)

hours = pd.Index(range(24), name="hour")
views = pd.Index([v for v in VIEW_COLORS if v in meta_df["view"].unique()], name="view")
full = (
    pd.MultiIndex.from_product([hours, views], names=["hour", "view"])
    .to_frame(index=False)
)
hour_counts = (
    full.merge(hour_counts, on=["hour", "view"], how="left")
        .fillna({"count": 0})
)
hour_counts["count"] = hour_counts["count"].astype(int)

fig = px.bar(
    hour_counts,
    x="hour", y="count",
    color="view",
    barmode="group",
    color_discrete_map=VIEW_COLORS,
    title="Acquisitions by hour of day"
)

fig.update_layout(
    xaxis=dict(dtick=1, title="Hour of Day"),
    yaxis_title="Count",
    bargap=0.1,
    bargroupgap=0.05,
    legend_title="",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    height=500,
)

fig.show()

fig.write_html("../assets/plots/hour_by_view.html", include_plotlyjs="cdn", full_html=False)


In [118]:

# --- inputs / orders ---
custom_scene_order = [
    'countryside', 'forest', 'water', 'skyscraper', 'mountain',
    'park', 'city', 'open field', 'industry', 'road',
    'office', 'conference room', 'corridor', 'bedroom', 'kitchen'
]
view_order = ["Outdoor", "Indoor w/ window", "Indoor w/o window"]

# --- aggregate counts like your matplotlib code ---
combo_counts = (
    meta_df.groupby(['scene', 'view', 'lighting'])
           .size().reset_index(name='count')
           .sort_values('count', ascending=False)
)
agg_counts = combo_counts.groupby(['scene', 'view'], as_index=False)['count'].sum()

# keep only scenes present in data but respect your custom order
scenes_in_data = [s for s in custom_scene_order if s in agg_counts['scene'].unique()]

# build full grid scene×view so missing combos show as 0 (keeps stacking consistent)
full = pd.DataFrame(list(product(scenes_in_data, view_order)), columns=['scene','view'])
plot_df = (full.merge(agg_counts, on=['scene','view'], how='left')
                .fillna({'count': 0}))
plot_df['count'] = plot_df['count'].astype(int)

# --- plotly stacked bar with inside labels ---
fig = px.bar(
    plot_df,
    x='scene', y='count', color='view',
    category_orders={'scene': scenes_in_data, 'view': view_order},
    color_discrete_map=VIEW_COLORS,
    title="Scene distribution by view",
    # text_auto=True,
)

fig.update_layout(
    barmode='stack',
    xaxis_title=None,
    yaxis_title='Count',
    legend_title='',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    height=500,
    bargap=0.15
)

fig.update_xaxes(tickangle=270)

fig.show()

fig.write_html("../assets/plots/scene_distribution_by_view.html", include_plotlyjs="cdn", full_html=False)

In [129]:

scene_order = sorted(meta_df["scene"].dropna().unique())
view_order = ["Outdoor", "Indoor w/ window", "Indoor w/o window"]

fig = px.box(
    meta_df.dropna(subset=["scene", "hour", "view"]),
    x="scene",
    y="hour",
    color="view",
    category_orders={"scene": custom_scene_order, "view": view_order},
    color_discrete_map=VIEW_COLORS,
    title="Acquisition times by scene category"
)

fig.update_layout(
    xaxis_title=None,
    yaxis_title="Hour of Day",
    legend_title="",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    height=500
)
fig.update_xaxes(tickangle=270)

fig.show()
fig.write_html("../assets/plots/acquisition_times_by_scene.html", include_plotlyjs="cdn", full_html=False)

In [113]:
views = ["Outdoor", "Indoor w/ window", "Indoor w/o window"]

# order weather by first appearance (like your code)
weather_order = meta_df["weather"].dropna().drop_duplicates().tolist()

# aggregate counts scene×view
weather_counts = (
    meta_df.groupby(["weather", "view"])
           .size().reset_index(name="count")
)

# build full grid so missing combos are zero (keeps stacks aligned)
full = (
    pd.MultiIndex.from_product([weather_order, views], names=["weather", "view"])
      .to_frame(index=False)
)
plot_df = (
    full.merge(weather_counts, on=["weather", "view"], how="left")
        .fillna({"count": 0})
)
plot_df["count"] = plot_df["count"].astype(int)

fig = px.bar(
    plot_df,
    x="weather", y="count",
    color="view",
    category_orders={"weather": weather_order, "view": views},
    color_discrete_map=VIEW_COLORS,
    title="Weather conditions by view",
    # text_auto=True
)

fig.update_layout(
    barmode="stack",
    xaxis_title=None,
    yaxis_title="Count",
    bargap=0.15,
    legend_title="",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    height=500,
)

fig.update_xaxes(tickangle=0)
fig.show()
# If your notebook lives in scripts/, save up one level:
fig.write_html("../assets/plots/weather_by_view.html", include_plotlyjs="cdn", full_html=False)


In [112]:
views = ["Outdoor", "Indoor w/ window", "Indoor w/o window"]

lighting_order = meta_df["lighting"].dropna().drop_duplicates().tolist()

lighting_counts = (
    meta_df.groupby(["lighting", "view"])
           .size().reset_index(name="count")
)

full = (
    pd.MultiIndex.from_product([lighting_order, views], names=["lighting", "view"])
      .to_frame(index=False)
)
plot_df = (
    full.merge(lighting_counts, on=["lighting", "view"], how="left")
        .fillna({"count": 0})
)
plot_df["count"] = plot_df["count"].astype(int)

fig = px.bar(
    plot_df,
    x="lighting", y="count",
    color="view",
    category_orders={"lighting": lighting_order, "view": views},
    color_discrete_map=VIEW_COLORS,
    title="Lighting conditions by view",
    # text_auto=True
)

fig.update_layout(
    barmode="stack",
    xaxis_title=None,
    yaxis_title="Count",
    bargap=0.15,
    legend_title="",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    height=500,
)

fig.update_xaxes(tickangle=0)
fig.show()
# If your notebook lives in scripts/, save up one level:
fig.write_html("../assets/plots/lighting_by_view.html", include_plotlyjs="cdn", full_html=False)


In [108]:
fig = px.histogram(
    meta_df,
    x="view",
    color="view",
    color_discrete_map=VIEW_COLORS,
    title="Distribution of view types",
    # text_auto=True
)

fig.update_layout(
    xaxis_title=None,
    yaxis_title="Count",
    bargap=0.15,
    legend_title="",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    height=450
)

fig.show()

fig.write_html("../assets/plots/view_distribution.html", include_plotlyjs="cdn", full_html=False)
